# Broadcasting Tutorial

Broadcasting is how PyTorch (and numpy) lets you do element-wise ops on tensors of *different* shapes, by virtually stretching the smaller one, without copying memory.

**The rule:** line up two shapes from the **right**. For each pair of dimensions:
- if they're equal, keep it
- if one of them is `1`, stretch it to match the other
- if a shape runs out of dimensions, pretend it has a `1` there
- if neither of the above holds, it's an error

Run the cells below in order.

In [ ]:
import torch

## 1. Same shape — no broadcasting needed

The baseline case: identical shapes just add element-wise.

In [ ]:
a = torch.tensor([[1, 2, 3],
                   [4, 5, 6]])
b = torch.tensor([[10, 20, 30],
                   [40, 50, 60]])
print(a.shape, b.shape)
print(a + b)

## 2. Scalar broadcasting

A scalar has shape `()` — every dimension is implicitly `1`, so it stretches to match anything.

In [ ]:
a = torch.tensor([[1, 2, 3],
                   [4, 5, 6]])
print(a + 100)

## 3. Vector + matrix (the common case)

`a` is `(2,3)`, `b` is `(3,)`. Align from the right: `b`'s missing dim is padded to `(1,3)`, then the `1` stretches to `2`. Result shape `(2,3)` — `b` gets added to *every row*.

In [ ]:
a = torch.tensor([[1, 2, 3],
                   [4, 5, 6]])          # shape (2,3)
b = torch.tensor([10, 20, 30])          # shape (3,)

print('a shape:', a.shape, 'b shape:', b.shape)
print(a + b)
print('result shape:', (a + b).shape)

Now flip it: a column vector `(2,1)` added to `(2,3)`. The `1` is in the *last* dim this time, so it stretches across columns instead — `c` gets added to *every column*.

In [ ]:
c = torch.tensor([[100], [200]])        # shape (2,1)
print(a + c)
print('result shape:', (a + c).shape)

## 4. `dim` and `keepdim` — the part that trips people up

`dim` picks **which axis gets collapsed** when you reduce (e.g. `.sum`). It is NOT a pointer to "the first element" — it names an axis.

For a 2D tensor of shape `(rows, cols)`:
- `dim=0` collapses the **row** axis → one number per **column** (a column sum)
- `dim=1` collapses the **column** axis → one number per **row** (a row sum)

`keepdim` controls whether the collapsed axis disappears (`False`, default) or stays as size `1` (`True`) in the output shape. The *numbers* are identical either way — only the shape differs, and that shape is what determines how the result broadcasts back against the original tensor.

In [ ]:
# small stand-in for the bigram count matrix P
counts = torch.tensor([[1, 2, 3],
                        [4, 0, 0],
                        [1, 1, 1]], dtype=torch.float32)
print(counts, '\n')

print('sum(dim=0, keepdim=False):', counts.sum(0, keepdim=False), counts.sum(0, keepdim=False).shape)
print('sum(dim=0, keepdim=True): ', counts.sum(0, keepdim=True),  counts.sum(0, keepdim=True).shape)
print()
print('sum(dim=1, keepdim=False):', counts.sum(1, keepdim=False), counts.sum(1, keepdim=False).shape)
print('sum(dim=1, keepdim=True): ', counts.sum(1, keepdim=True),  counts.sum(1, keepdim=True).shape)

### Why `keepdim=True` matters: row-normalizing

Say each row of `counts` should become a probability distribution (row sums to 1). You need to divide each row by *its own* row-total — that's `dim=1` (collapse columns → one total per row), kept as shape `(3,1)` so it broadcasts across columns correctly.

In [ ]:
row_sums = counts.sum(1, keepdim=True)   # shape (3,1)
probs = counts / row_sums
print(probs)
print('each row sums to 1:', probs.sum(1))

### The classic bug: wrong `dim`, or forgetting `keepdim`

Watch what happens if you use `dim=0` (column sums) instead — wrong axis, so rows don't sum to 1 anymore. And if you drop `keepdim`, the shape `(3,)` broadcasts against the *columns* rather than the rows, changing what gets divided by what.

In [ ]:
wrong_axis = counts / counts.sum(0, keepdim=True)   # normalizes columns, not rows
print('columns sum to 1 instead:', wrong_axis.sum(0))
print('rows do NOT sum to 1:', wrong_axis.sum(1), '\n')

no_keepdim = counts.sum(1, keepdim=False)   # shape (3,) instead of (3,1)
print('no_keepdim shape:', no_keepdim.shape)
# counts is (3,3), no_keepdim is (3,) -> right-aligned, broadcasts across the LAST dim (columns),
# which only happens to look reasonable here because the matrix is square (3x3).
print(counts / no_keepdim)

That last one is the dangerous case: with a square matrix, `(3,)` silently aligns against the wrong axis and still runs without error — it just gives the wrong answer. With a non-square matrix it would raise an error instead, which is actually the safer failure mode.

In [ ]:
rect = torch.rand(3, 5)          # 3 rows, 5 cols — not square
bad = rect.sum(1, keepdim=False)  # shape (3,)
try:
    rect / bad
except RuntimeError as e:
    print('RuntimeError:', e)

## 5. Higher-dimensional broadcasting

The same right-aligned rule works for any number of dimensions. `(5,1,4)` and `(1,3,4)` broadcast to `(5,3,4)` — each size-`1` axis stretches independently, giving an outer-product-like effect.

In [ ]:
x = torch.arange(5).view(5, 1, 1)   # shape (5,1,1)
y = torch.arange(3).view(1, 3, 1)   # shape (1,3,1)
z = x + y
print('z shape:', z.shape)
print(z.squeeze(-1))   # squeeze just for readable printing
# z[i,j,0] == i + j for every combination of i in 0..4 and j in 0..2

## 6. When it fails

Two shapes fail to broadcast when, at some aligned position, the sizes differ and neither is `1`.

In [ ]:
p = torch.rand(3, 4)
q = torch.rand(4, 3)
try:
    p + q
except RuntimeError as e:
    print('RuntimeError:', e)

## 7. Your turn

Using `logits` below (shape `(2, 27, 27)` — think: batch of 2, each a 27x27 matrix), write an expression that turns each `logits[b, i, :]` row into a probability distribution (softmax the last dimension using sum instead of exp/normalize, just to practice the shape mechanics).

In [ ]:
logits = torch.rand(2, 27, 27)

# TODO: normalize the last dimension so logits_norm[b, i, :].sum() == 1 for every b, i
logits_norm = ...

# uncomment to check:
# print(logits_norm.sum(-1))

<details>
<summary>Answer (click to expand)</summary>

```python
logits_norm = logits / logits.sum(-1, keepdim=True)
```

`logits.sum(-1, keepdim=True)` has shape `(2,27,1)`, which broadcasts against `(2,27,27)` by stretching across the last axis — exactly what you want when the thing you're dividing by is a per-row (last-dim) total.
</details>